# 04 — Feature Engineering

Goal: prepare two feature sets for modeling in the next notebook:

- **"Cheap" features** - things known from basic repo metadata alone 
  (stars, forks, language, license, age) without needing to deep-crawl 
  commit history, issues, or PRs for every repo
- **"Full" features** - cheap features + the deeper signals we collected 
  (commit velocity, contributor count, issue/PR patterns)

The comparison between these two models is the actual point of this 
phase: how much does the expensive data collection buy us over a 
lightweight screening approach? This also sidesteps a design trap - 
`risk_score`/`risk_category` were built directly from `days_since_last_commit`, 
`commits_per_month_90d`, `distinct_recent_commit_authors`, and `is_archived`, 
so training on those exact features would just have a model reverse-engineer 
our own formula rather than learn anything real. The cheap-feature model 
avoids this entirely; the full-feature model will need this kept in mind 
when interpreting results later.

In [9]:
!pip install scikit-learn

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean_labeled_repos.csv')
print(df.shape)

# "Cheap" - obtainable from basic repo metadata, no deep crawling needed
cheap_features = ['stars', 'forks', 'language', 'license', 'age_days']

# "Deep" - required the per-repo GraphQL queries we built in notebook 00
# Excluded from deep_features: commits_per_month_90d, commits_per_month_365d,
# and distinct_recent_commit_authors - these were DIRECT inputs to the
# risk_score formula in notebook 01, so including them let the model
# partially reconstruct the label rather than learn genuinely new signal
# (confirmed via feature importance: these 3 accounted for ~65% of a
# Random Forest's decisions, and drove a suspicious 1.00 F1 on the Low class)
deep_features = ['total_mentionable_users',
                  'open_issues_total', 'closed_issues_total',
                  'median_issue_resolution_days_recent',
                  'open_prs_total', 'merged_prs_total',
                  'median_pr_merge_days_recent', 'pr_merge_rate_recent',
                  'has_recent_pr_activity', 'has_recent_issue_activity']

target = 'risk_category'

print("Cheap features:", cheap_features)
print("Deep features:", deep_features)
print("Target:", target)

(498, 25)
Cheap features: ['stars', 'forks', 'language', 'license', 'age_days']
Deep features: ['total_mentionable_users', 'open_issues_total', 'closed_issues_total', 'median_issue_resolution_days_recent', 'open_prs_total', 'merged_prs_total', 'median_pr_merge_days_recent', 'pr_merge_rate_recent', 'has_recent_pr_activity', 'has_recent_issue_activity']
Target: risk_category


In [13]:
lang_counts = df['language'].value_counts()
print(lang_counts)
print()
print(f"Total distinct languages: {len(lang_counts)}")
print(f"Languages with <10 repos: {(lang_counts < 10).sum()}")

language
Python               92
JavaScript           68
TypeScript           60
None/Not detected    59
Go                   29
C++                  24
Rust                 17
Java                 17
C                    16
Shell                15
C#                   12
HTML                 11
Jupyter Notebook      9
Swift                 7
Kotlin                7
Ruby                  6
Dart                  5
PHP                   5
Objective-C           5
Markdown              3
Haskell               3
CSS                   3
MDX                   2
Lua                   2
Vim Script            2
Batchfile             1
Julia                 1
Clojure               1
Inno Setup            1
Astro                 1
MATLAB                1
Crystal               1
Elixir                1
Cairo                 1
Pascal                1
SCSS                  1
Vue                   1
Scala                 1
Groovy                1
Dockerfile            1
Solidity              1
HCL    

In [14]:
MIN_LANGUAGE_COUNT = 10

top_languages = lang_counts[lang_counts >= MIN_LANGUAGE_COUNT].index.tolist()
print("Languages kept as their own category:", top_languages)
print(f"({len(top_languages)} categories)")

df['language_grouped'] = df['language'].apply(lambda x: x if x in top_languages else 'Other')
print()
print(df['language_grouped'].value_counts())

Languages kept as their own category: ['Python', 'JavaScript', 'TypeScript', 'None/Not detected', 'Go', 'C++', 'Rust', 'Java', 'C', 'Shell', 'C#', 'HTML']
(12 categories)

language_grouped
Python               92
Other                78
JavaScript           68
TypeScript           60
None/Not detected    59
Go                   29
C++                  24
Rust                 17
Java                 17
C                    16
Shell                15
C#                   12
HTML                 11
Name: count, dtype: int64


In [15]:
license_counts = df['license'].value_counts()
print(license_counts)
print()
print(f"Total distinct licenses: {len(license_counts)}")

license
mit             182
No license       80
apache-2.0       75
other            56
gpl-3.0          32
agpl-3.0         20
bsd-3-clause     13
gpl-2.0           9
cc0-1.0           6
bsd-2-clause      5
cc-by-4.0         4
unlicense         4
isc               3
mpl-2.0           3
cc-by-sa-4.0      2
lgpl-3.0          2
ofl-1.1           1
0bsd              1
Name: count, dtype: int64

Total distinct licenses: 18


In [16]:
MIN_LICENSE_COUNT = 10

top_licenses = license_counts[license_counts >= MIN_LICENSE_COUNT].index.tolist()
print("Licenses kept as their own category:", top_licenses)

# named "Rare license" (not "Other") to avoid colliding with GitHub's own
# "other" license category, which means something different (has a license,
# just not a standard recognized one)
df['license_grouped'] = df['license'].apply(lambda x: x if x in top_licenses else 'Rare license')
print()
print(df['license_grouped'].value_counts())

Licenses kept as their own category: ['mit', 'No license', 'apache-2.0', 'other', 'gpl-3.0', 'agpl-3.0', 'bsd-3-clause']

license_grouped
mit             182
No license       80
apache-2.0       75
other            56
Rare license     40
gpl-3.0          32
agpl-3.0         20
bsd-3-clause     13
Name: count, dtype: int64


In [17]:
# One-hot encode the grouped categoricals
lang_dummies = pd.get_dummies(df['language_grouped'], prefix='lang')
license_dummies = pd.get_dummies(df['license_grouped'], prefix='license')

print(lang_dummies.shape, license_dummies.shape)
lang_dummies.head()

(498, 13) (498, 8)


,lang_C,lang_C#,lang_C++,lang_Go,lang_HTML,lang_Java,lang_JavaScript,lang_None/Not detected,lang_Other,lang_Python,lang_Rust,lang_Shell,lang_TypeScript
0,False,False,False,False,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,True,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,True
4,False,False,False,False,False,False,False,False,False,True,False,False,False


In [18]:
# Assemble cheap feature matrix: numeric cheap features + both dummy sets
cheap_numeric = df[['stars', 'forks', 'age_days']]
X_cheap = pd.concat([cheap_numeric, lang_dummies, license_dummies], axis=1)

# Assemble full feature matrix: cheap + deep numeric/boolean features
deep_numeric_bool = df[deep_features]
X_full = pd.concat([X_cheap, deep_numeric_bool], axis=1)

print("X_cheap shape:", X_cheap.shape)
print("X_full shape:", X_full.shape)

y = df['risk_category']
print("y shape:", y.shape)
print(y.value_counts())

X_cheap shape: (498, 24)
X_full shape: (498, 34)
y shape: (498,)
risk_category
Low       250
High      162
Medium     86
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split

X_cheap_train, X_cheap_test, y_train, y_test = train_test_split(
    X_cheap, y, test_size=0.2, random_state=42, stratify=y
)

# Full feature set uses the SAME split indices so both models are evaluated
# on identical train/test repos - makes the cheap-vs-full comparison fair
X_full_train = X_full.loc[X_cheap_train.index]
X_full_test = X_full.loc[X_cheap_test.index]

print("Train size:", X_cheap_train.shape[0])
print("Test size:", X_cheap_test.shape[0])
print()
print("Train class balance:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Test class balance:")
print(y_test.value_counts(normalize=True).round(3))

Train size: 398
Test size: 100

Train class balance:
risk_category
Low       0.503
High      0.324
Medium    0.173
Name: proportion, dtype: float64

Test class balance:
risk_category
Low       0.50
High      0.33
Medium    0.17
Name: proportion, dtype: float64


In [20]:
import joblib

joblib.dump({
    'X_cheap_train': X_cheap_train, 'X_cheap_test': X_cheap_test,
    'X_full_train': X_full_train, 'X_full_test': X_full_test,
    'y_train': y_train, 'y_test': y_test,
}, '../data/model_ready_data.pkl')

print("Saved model-ready data.")

Saved model-ready data.


## Feature Engineering Summary

- Grouped `language` (44 → 13 categories) and `license` (18 → 8 categories) 
  to avoid one-hot encoding dozens of near-empty columns on a 498-row dataset
- Built two feature sets to test a real question: how much does the deep, 
  expensive-to-collect data (commit velocity, contributor counts, issue/PR 
  patterns) actually improve predictions over cheap, easily-available 
  metadata alone (stars, forks, language, license, age)?
  - **Cheap features**: 24 columns
  - **Full features**: 37 columns (cheap + 13 deep signals)
- Target is `risk_category` (Low/Medium/High), deliberately *not* 
  `is_archived` for the deep-feature model, since risk_category's 
  construction directly used several deep features - a model predicting 
  it from those same features would just reverse-engineer the scoring 
  formula rather than learn anything new. The cheap-vs-full comparison 
  is what makes this a real predictive question rather than a 
  circular one.
- Stratified 80/20 train/test split (398/100), class balance preserved: 
  ~50% Low, ~33% High, ~17% Medium in both sets

**Next:** train and compare models (starting with Logistic Regression as 
baseline, then Random Forest) on both feature sets, evaluate with 
precision/recall/F1 rather than raw accuracy given the class imbalance, 
and use explainability (feature importance / SHAP) to see whether cheap 
features alone can catch most of the signal.